In [ ]:
import json
import re
import emoji
from googletrans import Translator
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load slang dictionary (example)
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "tdk": "tidak",
    "aja": "saja",
    # Add more slang words as needed
}

def replace_slang(text, slang_dict):
    words = text.split()
    return ' '.join([slang_dict.get(w, w) for w in words])

def remove_extra_chars(text):
    text = re.sub(r'(.)\1{2,}', r'\1', text)  # Remove repeated chars (>2)
    return text

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def remove_usernames(text):
    return re.sub(r'@\w+', '', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

def preprocess_text(text, stop_words, slang_dict):
    # 1. Back translation
    translator = Translator()
    try:
        en_text = translator.translate(text, src='id', dest='en').text
        text = translator.translate(en_text, src='en', dest='id').text
    except Exception:
        pass  # If translation fails, keep original

    # 2. Remove stopwords
    tokens = word_tokenize(text)
    text = ' '.join([w for w in tokens if w.lower() not in stop_words])

    # 3. Replace slang words
    text = replace_slang(text, slang_dict)

    # 4. Remove extra characters
    text = remove_extra_chars(text)

    # 5. Convert emojis to phrases
    text = convert_emojis(text)

    # 6. Remove usernames
    text = remove_usernames(text)

    # 7. Remove numbers
    text = remove_numbers(text)

    # 8. Remove punctuation and convert to lowercase
    text = remove_punctuation(text).lower()

    return text.strip()

# Load stopwords
import nltk
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('indonesian'))

# Load data
with open('Preprocessing/fetched_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Assume data is a list of dicts with a 'text' field
for item in data:
    item['preprocessed_text'] = preprocess_text(item['text'], stop_words, slang_dict)

# Save preprocessed data
with open('Preprocessing/fetched_data_preprocessed.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
